# Throat pruning experiment

What happens to `tau_sq_c`, `C_c`, `effective_conductance`, etc. if we remove "problematic" throats from a pore network, recompute Berg effective geometry (volumes/lengths/areas/coordination numbers), and re-solve?

Everything you'd normally tune lives in **Cell 3 (config)** and **Cell 4 (criterion)** below. You shouldn't need to open `analysis/artificial/throat_removal.py` again unless you want a rule that doesn't fit the `rank_criterion(...)` / plain-function patterns shown here.

Available throat-table columns to build criteria from (see `result["throat_table"].columns` for the full list):

| column | meaning |
|---|---|
| `C` | constriction ratio, `r_throat / min(r_body1, r_body2)` |
| `C_geomean` | constriction ratio, `r_throat / sqrt(r_body1 * r_body2)` (narrow relative to *both* neighbours) |
| `f` | fraction of total network current carried by the throat |
| `G` | throat conductance |
| `R1`, `R_throat`, `R2`, `R_edge` | series resistances (body1 segment, throat channel, body2 segment, total) |
| `P_edge_frac` | share of total dissipated power in the *whole* body1-throat-body2 segment (I^2 * R_edge) |
| `P_throat_only_frac` | share of total dissipated power in the throat channel *alone* (I^2 * R_throat) |
| `z_body1`, `z_body2` | coordination number of each endpoint body |
| `throat_id` | the id to map back to `network.throats` |

In [1]:
# Cell 1 -- imports & path setup (run once)
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()  # analysis/notebooks -> analysis -> repo root
print(f"Repo root: {REPO_ROOT}")
sys.path.insert(0, str(REPO_ROOT / "analysis" / "artificial"))

from throat_removal import prune_and_recompute, rank_criterion

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

Repo root: /trace/group/acmegroup/rochan/Constrictivity


## Sweeping / comparing multiple criteria

To compare several criteria at once, just loop and collect results -- each call re-solves independently, so no state leaks between runs.

In [8]:
# Cell 8 -- optional: compare several criteria in one table
criteria_to_try = {
    "bottleneck_default": rank_criterion("C_geomean", 10, "bottom", "P_throat_only_frac", 10, "top"),
    "bottom5pct_C": rank_criterion("C", 5, "bottom"),
    "top10pct_P_edge": rank_criterion("P_edge_frac", 10, "top"),
}

rows = []
for name, crit in criteria_to_try.items():
    r = prune_and_recompute(body_array, segmented, config, criterion=crit, verbose=False)
    row = {"criterion": name, "n_removed": r["n_throats_removed"], "solve_failed": r["solve_failed"] is not None}
    if r["pruned"] is not None:
        row.update({f"delta_{k}": v for k, v in r["delta"].items()
                    if k in ("tau_sq_c", "C_c", "effective_conductance", "phi_c", "Omega_c")})
    rows.append(row)

pd.DataFrame(rows)

NameError: name 'body_array' is not defined

## Running on multiple microstructures

`data/` currently has two microstructures with the raw `body_array.npy` + a segmented volume that `prune_and_recompute` needs (it rebuilds the network from scratch, so cached `TRACE/*.csv` summaries alone aren't enough):

| microstructure | body_array shape | bodies | direction | surface_axis | segmented file |
|---|---|---|---|---|---|
| `microstructure1` | 96³ | 404 | `x` | `0` | `segmented.npy` |
| `Coarse_microstructure_0` | 150³ | 37 | `y` | `1` | `segmented_premesh.npy` |

(configs read from each microstructure's `data/<name>/version*/berg_cc/config.json` and `manifest.json` -- re-check those if you add a new microstructure, since `direction`/`surface_axis`/`segmented_file` differ per dataset.)

`run_on_microstructures(...)` runs `prune_and_recompute` once per microstructure with the *same* criterion and returns one tidy summary DataFrame (`baseline_<metric>`, `pruned_<metric>`, `delta_<metric>` columns) plus a dict of the full per-microstructure results for follow-up.

In [2]:
# Cell 9 -- define the microstructure set once (edit configs here if you add more)
from throat_removal import run_on_microstructures

In [3]:
import json

microstructures = {}
for ms_dir in sorted((REPO_ROOT / "data").iterdir()):
    if not (ms_dir / "latest_version.json").exists():
        continue  # skip loose files like YSZ_cleaned_x.npy
    manifest = json.loads((ms_dir / "latest_version.json").read_text())
    seg_file = manifest.get("segmented_file", "segmented_premesh.npy")

    # # find the latest berg_cc config actually used
    # config_paths = sorted((ms_dir).glob("version*/berg_cc/config.json"))
    # if not config_paths:
    #     continue
    # berg_config = json.loads(config_paths[-1].read_text())

    microstructures[ms_dir.name] = {
        "body_array": np.load(ms_dir / "body_array.npy"),
        "segmented": np.load(ms_dir / "segmented_premesh.npy"),
        "config": {
        "min_throat_size": 5,
        "dilate_both": True,
        "surface_axis": 1,
        "skip_surface_surface": True,
        "area_method": "vox_projection",
        "alpha": 0.5,
        "split_volume_equal": True,
        "sigma": 1.0,
        "n_jobs": 8,
        "inlet_value": 1.0,
        "outlet_value": 0.0,
        "direction": "y",
        "delta_V": 1.0,
        "voxel_size": 1.0,
        "streamtube_method": "greedy_fast"
    },
    }

for name, ms in microstructures.items():
    print(f"{name}: body_array {ms['body_array'].shape}, {ms['body_array'].max()} bodies")


Coarse_microstructure_0: body_array (150, 150, 150), 37 bodies
Coarse_microstructure_1: body_array (150, 150, 150), 48 bodies
Coarse_microstructure_3: body_array (150, 150, 150), 58 bodies
Coarse_microstructure_4: body_array (150, 150, 150), 64 bodies
Coarse_microstructure_5: body_array (150, 150, 150), 59 bodies
Coarse_microstructure_6: body_array (150, 150, 150), 45 bodies
Coarse_microstructure_7: body_array (150, 150, 150), 64 bodies
Coarse_microstructure_8: body_array (150, 150, 150), 82 bodies
Coarse_microstructure_9: body_array (150, 150, 150), 74 bodies
Fine_microstructure_0: body_array (150, 150, 150), 316 bodies
Fine_microstructure_1: body_array (150, 150, 150), 357 bodies
Fine_microstructure_2: body_array (150, 150, 150), 379 bodies
Fine_microstructure_3: body_array (150, 150, 150), 383 bodies
Fine_microstructure_4: body_array (150, 150, 150), 364 bodies
Fine_microstructure_5: body_array (150, 150, 150), 364 bodies
Fine_microstructure_6: body_array (150, 150, 150), 359 bodies

In [4]:
# Cell 10 -- run the default criterion across all microstructures and collect results
from throat_removal import with_before_after_columns

In [5]:
fine_micro = {k:v for k,v in microstructures.items() if "Fine" in k}

for name, ms in fine_micro.items():
    print(f"{name}: body_array {ms['body_array'].shape}, {ms['body_array'].max()} bodies")

Fine_microstructure_0: body_array (150, 150, 150), 316 bodies
Fine_microstructure_1: body_array (150, 150, 150), 357 bodies
Fine_microstructure_2: body_array (150, 150, 150), 379 bodies
Fine_microstructure_3: body_array (150, 150, 150), 383 bodies
Fine_microstructure_4: body_array (150, 150, 150), 364 bodies
Fine_microstructure_5: body_array (150, 150, 150), 364 bodies
Fine_microstructure_6: body_array (150, 150, 150), 359 bodies
Fine_microstructure_7: body_array (150, 150, 150), 380 bodies
Fine_microstructure_8: body_array (150, 150, 150), 360 bodies
Fine_microstructure_9: body_array (150, 150, 150), 382 bodies


In [6]:
# Cell 11 -- sweep a criterion across BOTH microstructures and multiple percentiles at once,
# to see whether a trend (e.g. "more removed -> worse tortuosity/constriction") holds
# consistently across datasets, not just one.
sweep_rows = []
for pct in [10, 15, 20]:
    crit = rank_criterion("C_geomean", pct, "bottom", "P_throat_only_frac", pct, "top")
    df_pct, _ = run_on_microstructures(fine_micro, criterion=crit, verbose=False)
    df_pct.insert(0, "percentile", pct)
    sweep_rows.append(df_pct)

sweep_df = pd.concat(sweep_rows, ignore_index=True)
sweep_display_df = with_before_after_columns(
    sweep_df, ["inv_tau_sq_c", "inv_C_c", "inv_F_c", "phi_c","effective_conductance"]
)
sweep_display_df[[
    "percentile", "ms_name", "solve_failed", "n_throats_removed",
    "inv_tau_sq_c_before/after", "inv_C_c_before/after", "inv_F_c_before/after", "phi_c_before/after",
]]

[build_network_arrays] 269 throat(s) used uniform-pipe fallback.
[build_network_arrays] 316 bodies, 601 throats indexed.
Current threshold I_threshold = 1.000e-10


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


[build_network_arrays] 266 throat(s) used uniform-pipe fallback.
[build_network_arrays] 316 bodies, 594 throats indexed.
Current threshold I_threshold = 1.000e-10


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


[build_network_arrays] 309 throat(s) used uniform-pipe fallback.
[build_network_arrays] 357 bodies, 673 throats indexed.
Current threshold I_threshold = 1.000e-10
[build_network_arrays] 306 throat(s) used uniform-pipe fallback.
[build_network_arrays] 357 bodies, 665 throats indexed.
Current threshold I_threshold = 1.000e-10


...

[build_network_arrays] 253 throat(s) used uniform-pipe fallback.
[build_network_arrays] 360 bodies, 601 throats indexed.
Current threshold I_threshold = 1.000e-10
[build_network_arrays] 241 throat(s) used uniform-pipe fallback.
[build_network_arrays] 360 bodies, 553 throats indexed.
Current threshold I_threshold = 1.000e-10


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


[build_network_arrays] 329 throat(s) used uniform-pipe fallback.
[build_network_arrays] 382 bodies, 719 throats indexed.
Current threshold I_threshold = 9.750e-11
[build_network_arrays] 318 throat(s) used uniform-pipe fallback.
[build_network_arrays] 382 bodies, 678 throats indexed.
Current threshold I_threshold = 9.930e-11


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


,percentile,ms_name,solve_failed,n_throats_removed,inv_tau_sq_c_before/after,inv_C_c_before/after,inv_F_c_before/after,phi_c_before/after
0,10,Fine_microstructure_0,False,7,2.599/2.702,0.6062/0.6179,0.09648/0.0918,0.4137/0.4015
1,10,Fine_microstructure_1,False,8,2.747/2.833,0.6047/0.5957,0.08343/0.0797,0.379/0.379
2,10,Fine_microstructure_2,False,8,3.324/3.489,0.5731/0.5637,0.06151/0.05763,0.3568/0.3567
3,10,Fine_microstructure_3,False,10,2.754/2.844,0.5703/0.5632,0.07092/0.06775,0.3424/0.3421
4,10,Fine_microstructure_4,False,14,2.973/3.112,0.6021/0.5819,0.07598/0.06944,0.3751/0.3713
5,10,Fine_microstructure_5,False,7,3.027/3.112,0.5603/0.5586,0.06236/0.05896,0.3368/0.3285
6,10,Fine_microstructure_6,False,13,2.918/3.132,0.6411/0.6275,0.08452/0.07631,0.3847/0.3809
7,10,Fine_microstructure_7,False,16,3.121/3.666,0.5557/0.5425,0.06316/0.05099,0.3546/0.3446
8,10,Fine_microstructure_8,False,14,2.928/3.107,0.6068/0.6082,0.07268/0.0686,0.3507/0.3504
9,10,Fine_microstructure_9,False,7,2.944/3.005,0.6297/0.6297,0.07721/0.07515,0.361/0.3586


In [8]:
output_df = sweep_display_df[[
    "percentile", "ms_name", "solve_failed", "n_throats_removed",
    "inv_tau_sq_c_before/after", "inv_C_c_before/after", "inv_F_c_before/after", "phi_c_before/after",
]]

In [10]:
output_df.to_csv(REPO_ROOT / "analysis" / "artificial" / "throat_pruning_sweep_results.csv", index=False)

In [6]:
# Cell 11 -- sweep a criterion across BOTH microstructures and multiple percentiles at once,
# to see whether a trend (e.g. "more removed -> worse tortuosity/constriction") holds
# consistently across datasets, not just one.
from throat_removal import *

sweep_rows_random = []
for pct in [10, 15, 20]:
    crit = rank_criterion("C_geomean", pct, "bottom", "P_throat_only_frac", pct, "top")
    # crit = random_criterion(percentile=pct,seed=None)
    random_matched = random_criterion(match=crit, seed=0)
    df_pct, _ = run_on_microstructures(fine_micro, criterion=random_matched, verbose=False)
    df_pct.insert(0, "percentile", pct)
    sweep_rows_random.append(df_pct)

sweep_df_random = pd.concat(sweep_rows_random, ignore_index=True)
sweep_display_df_random = with_before_after_columns(
    sweep_df_random, ["inv_tau_sq_c", "inv_C_c", "inv_F_c", "phi_c","effective_conductance"]
)
sweep_display_df_random[[
    "percentile", "ms_name", "solve_failed", "n_throats_removed",
    "inv_tau_sq_c_before/after", "inv_C_c_before/after", "inv_F_c_before/after", "phi_c_before/after",
]]

[build_network_arrays] 269 throat(s) used uniform-pipe fallback.
[build_network_arrays] 316 bodies, 601 throats indexed.
Current threshold I_threshold = 1.000e-10


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


[build_network_arrays] 263 throat(s) used uniform-pipe fallback.
[build_network_arrays] 316 bodies, 594 throats indexed.
Current threshold I_threshold = 1.000e-10


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


[build_network_arrays] 309 throat(s) used uniform-pipe fallback.
[build_network_arrays] 357 bodies, 673 throats indexed.
Current threshold I_threshold = 1.000e-10
[build_network_arrays] 302 throat(s) used uniform-pipe fallback.
[build_network_arrays] 357 bodies, 665 throats indexed.
Current threshold I_threshold = nan


...

[build_network_arrays] 253 throat(s) used uniform-pipe fallback.
[build_network_arrays] 360 bodies, 601 throats indexed.
Current threshold I_threshold = 1.000e-10
[build_network_arrays] 202 throat(s) used uniform-pipe fallback.
[build_network_arrays] 360 bodies, 553 throats indexed.
Current threshold I_threshold = 1.000e-10


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


[build_network_arrays] 329 throat(s) used uniform-pipe fallback.
[build_network_arrays] 382 bodies, 719 throats indexed.
Current threshold I_threshold = 9.750e-11
[build_network_arrays] 295 throat(s) used uniform-pipe fallback.
[build_network_arrays] 382 bodies, 678 throats indexed.
Current threshold I_threshold = 9.658e-11


/trace/group/acmegroup/conda_envs/envs/rbenv/lib/python3.10/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


,percentile,ms_name,solve_failed,n_throats_removed,inv_tau_sq_c_before/after,inv_C_c_before/after,inv_F_c_before/after,phi_c_before/after
0,10,Fine_microstructure_0,False,7,2.599/2.594,0.6062/0.6064,0.09648/0.09616,0.4137/0.4114
1,10,Fine_microstructure_1,True,8,2.747/failed,0.6047/failed,0.08343/failed,0.379/failed
2,10,Fine_microstructure_2,False,8,3.324/3.369,0.5731/0.581,0.06151/0.06149,0.3568/0.3566
3,10,Fine_microstructure_3,False,10,2.754/2.786,0.5703/0.5694,0.07092/0.06916,0.3424/0.3384
4,10,Fine_microstructure_4,True,14,2.973/failed,0.6021/failed,0.07598/failed,0.3751/failed
5,10,Fine_microstructure_5,False,7,3.027/3,0.5603/0.5512,0.06236/0.06159,0.3368/0.3352
6,10,Fine_microstructure_6,False,13,2.918/2.956,0.6411/0.6315,0.08452/0.08188,0.3847/0.3832
7,10,Fine_microstructure_7,False,16,3.121/3.141,0.5557/0.556,0.06316/0.06241,0.3546/0.3525
8,10,Fine_microstructure_8,False,14,2.928/2.965,0.6068/0.6125,0.07268/0.07208,0.3507/0.3489
9,10,Fine_microstructure_9,False,7,2.944/2.973,0.6297/0.6285,0.07721/0.0762,0.361/0.3605


In [8]:
sweep_display_df_random[[
    "percentile", "ms_name", "solve_failed","baseline_n_throats","n_throats_removed",
    "inv_tau_sq_c_before/after", "inv_C_c_before/after", "inv_F_c_before/after", "phi_c_before/after",
]]

,percentile,ms_name,solve_failed,baseline_n_throats,n_throats_removed,inv_tau_sq_c_before/after,inv_C_c_before/after,inv_F_c_before/after,phi_c_before/after
0,10,Fine_microstructure_0,False,601,7,2.599/2.594,0.6062/0.6064,0.09648/0.09616,0.4137/0.4114
1,10,Fine_microstructure_1,True,673,8,2.747/failed,0.6047/failed,0.08343/failed,0.379/failed
2,10,Fine_microstructure_2,False,664,8,3.324/3.369,0.5731/0.581,0.06151/0.06149,0.3568/0.3566
3,10,Fine_microstructure_3,False,624,10,2.754/2.786,0.5703/0.5694,0.07092/0.06916,0.3424/0.3384
4,10,Fine_microstructure_4,True,641,14,2.973/failed,0.6021/failed,0.07598/failed,0.3751/failed
5,10,Fine_microstructure_5,False,620,7,3.027/3,0.5603/0.5512,0.06236/0.06159,0.3368/0.3352
6,10,Fine_microstructure_6,False,679,13,2.918/2.956,0.6411/0.6315,0.08452/0.08188,0.3847/0.3832
7,10,Fine_microstructure_7,False,658,16,3.121/3.141,0.5557/0.556,0.06316/0.06241,0.3546/0.3525
8,10,Fine_microstructure_8,False,601,14,2.928/2.965,0.6068/0.6125,0.07268/0.07208,0.3507/0.3489
9,10,Fine_microstructure_9,False,719,7,2.944/2.973,0.6297/0.6285,0.07721/0.0762,0.361/0.3605


In [9]:
random_df = sweep_display_df_random[[
    "percentile", "ms_name", "solve_failed","baseline_n_throats","n_throats_removed",
    "inv_tau_sq_c_before/after", "inv_C_c_before/after", "inv_F_c_before/after", "phi_c_before/after",
]]

In [10]:
random_df.to_csv(REPO_ROOT / "analysis" / "artificial" / "throat_pruning_random_sweep_results.csv", index=False)